# Deterministic hard coal NPV

Calculate a single hard coal NPV case using base values for technology triangular distributions, mean values for market-price scaled beta distributions, midpoint values for uniform distributions, and fixed values for deterministic parameters.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from electricity_model import (
    calculate_capacity_kw,
    calculate_capacity_mw,
    calculate_level_cash_flow_present_value_factor,
)
from electricity_parameters import (
    ANNUAL_ELECTRICITY_OUTPUT_MWH,
    HARD_COAL_CAPEX_DISTRIBUTION,
    HARD_COAL_EMISSIONS_DISTRIBUTION,
    HARD_COAL_FIXED_OPEX_DISTRIBUTION,
    HARD_COAL_FUEL_CONSUMPTION_DISTRIBUTION,
    HARD_COAL_FULL_LOAD_HOURS,
    HARD_COAL_VARIABLE_OPEX_DISTRIBUTION,
    LIFETIME_ELECTRICITY_YEARS,
    RETAIL_PRICE_ELECTRICITY_EUR_PER_MWH,
)
from general_parameters import (
    CARBON_PRICE_EUR_PER_T,
    COAL_PRICE_DISTRIBUTION,
    INTEREST_RATE,
)

In [2]:
annual_output_mwh = ANNUAL_ELECTRICITY_OUTPUT_MWH.value
full_load_hours = HARD_COAL_FULL_LOAD_HOURS.value
capacity_mw = calculate_capacity_mw(annual_output_mwh, full_load_hours)
capacity_kw = calculate_capacity_kw(annual_output_mwh, full_load_hours)

capex_eur_per_kw = (
    HARD_COAL_CAPEX_DISTRIBUTION.lower_bound
    + HARD_COAL_CAPEX_DISTRIBUTION.upper_bound
) / 2
fixed_opex_eur_per_kw_year = HARD_COAL_FIXED_OPEX_DISTRIBUTION.mode
variable_opex_eur_per_mwh = HARD_COAL_VARIABLE_OPEX_DISTRIBUTION.mode
fuel_consumption_mwh_th_per_mwh_e = HARD_COAL_FUEL_CONSUMPTION_DISTRIBUTION.mode
emissions_tco2_per_mwh_e = HARD_COAL_EMISSIONS_DISTRIBUTION.mode
coal_price_eur_per_mwh_th = COAL_PRICE_DISTRIBUTION.mean
electricity_price_eur_per_mwh = RETAIL_PRICE_ELECTRICITY_EUR_PER_MWH.value
carbon_price_eur_per_t = CARBON_PRICE_EUR_PER_T.value
lifetime_years = int(LIFETIME_ELECTRICITY_YEARS.value)
discount_rate = INTEREST_RATE.value

In [3]:
initial_capex_eur = capacity_kw * capex_eur_per_kw
annual_revenue_eur = annual_output_mwh * electricity_price_eur_per_mwh
annual_fixed_opex_eur = capacity_kw * fixed_opex_eur_per_kw_year
annual_variable_opex_eur = annual_output_mwh * variable_opex_eur_per_mwh
annual_fuel_cost_eur = (
    annual_output_mwh
    * fuel_consumption_mwh_th_per_mwh_e
    * coal_price_eur_per_mwh_th
)
annual_emissions_cost_eur = (
    annual_output_mwh * emissions_tco2_per_mwh_e * carbon_price_eur_per_t
)
annual_net_cash_flow_eur = (
    annual_revenue_eur
    - annual_fixed_opex_eur
    - annual_variable_opex_eur
    - annual_fuel_cost_eur
    - annual_emissions_cost_eur
)
present_value_factor = calculate_level_cash_flow_present_value_factor(
    lifetime_years=lifetime_years,
    discount_rate=discount_rate,
)
npv_eur = -initial_capex_eur + annual_net_cash_flow_eur * present_value_factor

pd.Series(
    {
        "annual_output_mwh": annual_output_mwh,
        "capacity_mw": capacity_mw,
        "capacity_kw": capacity_kw,
        "capex_eur_per_kw": capex_eur_per_kw,
        "fixed_opex_eur_per_kw_year": fixed_opex_eur_per_kw_year,
        "variable_opex_eur_per_mwh": variable_opex_eur_per_mwh,
        "fuel_consumption_mwh_th_per_mwh_e": fuel_consumption_mwh_th_per_mwh_e,
        "coal_price_eur_per_mwh_th": coal_price_eur_per_mwh_th,
        "emissions_tco2_per_mwh_e": emissions_tco2_per_mwh_e,
        "electricity_price_eur_per_mwh": electricity_price_eur_per_mwh,
        "carbon_price_eur_per_t": carbon_price_eur_per_t,
        "lifetime_years": lifetime_years,
        "discount_rate": discount_rate,
        "present_value_factor": present_value_factor,
        "initial_capex_million_eur": initial_capex_eur / 1_000_000,
        "annual_revenue_million_eur": annual_revenue_eur / 1_000_000,
        "annual_fixed_opex_million_eur": annual_fixed_opex_eur / 1_000_000,
        "annual_variable_opex_million_eur": annual_variable_opex_eur / 1_000_000,
        "annual_fuel_cost_million_eur": annual_fuel_cost_eur / 1_000_000,
        "annual_emissions_cost_million_eur": annual_emissions_cost_eur / 1_000_000,
        "annual_net_cash_flow_million_eur": annual_net_cash_flow_eur / 1_000_000,
        "npv_million_eur": npv_eur / 1_000_000,
    }
)

annual_output_mwh                    1000000.000000
capacity_mw                              243.902439
capacity_kw                           243902.439024
capex_eur_per_kw                        2000.000000
fixed_opex_eur_per_kw_year                37.000000
variable_opex_eur_per_mwh                  5.000000
fuel_consumption_mwh_th_per_mwh_e          2.560000
coal_price_eur_per_mwh_th                 12.110000
emissions_tco2_per_mwh_e                   0.870000
electricity_price_eur_per_mwh             94.070000
carbon_price_eur_per_t                     1.000000
lifetime_years                            25.000000
discount_rate                              0.080000
present_value_factor                      10.674776
initial_capex_million_eur                487.804878
annual_revenue_million_eur                94.070000
annual_fixed_opex_million_eur              9.024390
annual_variable_opex_million_eur           5.000000
annual_fuel_cost_million_eur              31.001600
annual_emiss